### Structured Output — Getting Reliable Data from an LLM


In [25]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

In [26]:
# Step 1: Define the shape you want FIRST (from earlier)
from pydantic import BaseModel
from google.genai import types

# This class IS the shape we want back. Note the types: str, int, List of str.
class Recipe(BaseModel):
    title: str
    ingredients: list[str]
    calories: int
    prep_time_minutes: int

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Give me a simple Pakistani dal recipe.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",    #response_mime_type="application/json" → tells the AI to reply in JSON format (structured data), not casual paragraph text
        response_schema=Recipe,                  #response_schema=Recipe → tells it to match your Recipe class's exact structure
        ),
                 
)
print(response.text)

{"title": "Simple Pakistani Tadka Dal", "ingredients": ["1 cup red lentils (masoor dal)", "1/2 cup yellow split moong dal", "1 medium onion, finely chopped", "2 tomatoes, chopped", "1 tablespoon ginger-garlic paste", "1 teaspoon turmeric powder", "1 teaspoon red chili powder", "1 teaspoon cumin seeds", "2 tablespoons ghee or vegetable oil", "3 cloves garlic, sliced", "2 dried red chilies", "Fresh cilantro, chopped for garnish", "Salt to taste", "4 cups water"], "calories": 280, "prep_time_minutes": 10}


In [27]:
# .parsed gives you back an actual validated Recipe object (not just text!)
response.parsed

Recipe(title='Simple Pakistani Tadka Dal', ingredients=['1 cup red lentils (masoor dal)', '1/2 cup yellow split moong dal', '1 medium onion, finely chopped', '2 tomatoes, chopped', '1 tablespoon ginger-garlic paste', '1 teaspoon turmeric powder', '1 teaspoon red chili powder', '1 teaspoon cumin seeds', '2 tablespoons ghee or vegetable oil', '3 cloves garlic, sliced', '2 dried red chilies', 'Fresh cilantro, chopped for garnish', 'Salt to taste', '4 cups water'], calories=280, prep_time_minutes=10)

In [28]:
recipe = response.parsed
print(recipe.title)
print(recipe.calories)
print(len(recipe.ingredients))

Simple Pakistani Tadka Dal
280
14


### Exercise: Customer Review Analyzer

In [30]:
from pydantic import BaseModel
from google.genai import types

class ReviewAnalysis(BaseModel):
    sentiment:str
    rating:int
    pros:list[str]
    cons:list[str]

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Customer Review Analyzer",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",    
        response_schema=ReviewAnalysis,                 
        ),
                 
)
print(response.text)

{"sentiment":"Positive","rating":4,"pros":["Excellent sound quality","Long battery life","Effective noise cancellation"],"cons":["Tight fit on larger heads","Bulky carrying case"]}


In [39]:
analysis = response.parsed
print(f"sentiment:{analysis.sentiment}")
print(f"Rating: {analysis.rating}/5")

print("\npros:")
for pro in analysis.pros:
    print(f"- {pro}")

print("\ncons:")
for con in analysis.cons:
    print(f"- {con}")

sentiment:Positive
Rating: 4/5

pros:
- Excellent sound quality
- Long battery life
- Effective noise cancellation

cons:
- Tight fit on larger heads
- Bulky carrying case


### Bonus 1: Analyze 3 different reviews

In [ ]:
reviews = [
    "I ordered the wireless earbuds last week. Sound quality is amazing and battery lasts all day, but the case feels cheap and it arrived two days late. Probably won't buy from this brand again.",
    "This blender is fantastic! Powerful motor, easy to clean, and it crushed ice like nothing. Highly recommend to anyone who makes smoothies daily.",
    "Terrible experience. The product arrived broken, customer service took a week to respond, and I still haven't gotten a refund. Would not recommend."
]

for review in reviews:
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=f"Analyze this customer review: {review}",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ReviewAnalysis,
        ),
    )
    analysis = response.parsed
    print(f"{analysis.sentiment} ({analysis.rating}/5)")

### Bonus 2: Add a recommended_action field

In [47]:
class ReviewAnalysis(BaseModel):
    sentiment:str
    rating:int
    pros:list[str]
    cons:list[str]
    recommended_action: str

reviews = [
    "I ordered the wireless earbuds last week. Sound quality is amazing and battery lasts all day, but the case feels cheap and it arrived two days late. Probably won't buy from this brand again.",
    "This blender is fantastic! Powerful motor, easy to clean, and it crushed ice like nothing. Highly recommend to anyone who makes smoothies daily.",
    "Terrible experience. The product arrived broken, customer service took a week to respond, and I still haven't gotten a refund. Would not recommend."
]

for review in reviews:
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=f"Analyze this customer review: {review}",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ReviewAnalysis,
        ),
    )
    analysis = response.parsed
    print(f"{analysis.sentiment} ({analysis.rating}/5)")
    print(f"Recommended Action: {analysis.recommended_action}")


mixed (2/5)
Recommended Action: Improve the material quality of the charging case and address logistics delays to ensure on-time delivery.
positive (5/5)
Recommended Action: Highlight this blender in marketing campaigns focused on daily smoothie makers, emphasizing its powerful motor and ease of cleaning.
Negative (1/5)
Recommended Action: Issue a full refund immediately, reach out to the customer with an apology, and investigate shipping package durability.
